In [1]:
import models 
import utils
import wbic

import numpy as np
import pymc as pm
import numpy as np
import pytensor.tensor as pt
import arviz as az

np.random.seed(42)
X = np.random.normal(loc=0, scale=1, size=500)
model = models.tempered_gaussian_mixture(
    X,
    n_components=2,
    beta = 1/np.log(X.size),
)
gmm = wbic.BayesianModel(
    model=model,
    observations=X,
)
gmm.sample(
    draws=2000,
    tune=1000,
    chains=4,
    #max_treedepth=50,
    #target_accept=.995
)
gmm.WBIC()
LC = gmm.learning_coefficient()
print(f'estimated WBIC: {gmm.WBIC()}')
print(f'estimated LC: {LC}')

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [weights, mus, sigmas]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 10 seconds.
There were 21 divergences after tuning. Increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


estimated WBIC: -707.7618296172473
estimated LC: 1.5919024698953714


In [2]:
LC = gmm.learning_coefficient(method='empirical_loss')
print(f'estimated LC: {LC}')

estimated LC: 1.1861578507291632


In [8]:
idata = gmm.inference_data
idata.add_groups(
    log_likelihood={"log_likelihood":idata.posterior['log_likelihood'].values}
)
idata
#az.loo(idata)

ValueError: ['log_likelihood'] group(s) already exists.

In [4]:
idata.add_groups(
    log_likelihood={idata.posterior['log_likelihood']}
)

<xarray.Dataset> Size: 32MB
Dimensions:               (chain: 4, draw: 2000, mus_dim_0: 2,
                           weights_dim_0: 2, sigmas_dim_0: 2,
                           log_likelihood_dim_0: 500)
Coordinates:
  * chain                 (chain) int64 32B 0 1 2 3
  * draw                  (draw) int64 16kB 0 1 2 3 4 ... 1996 1997 1998 1999
  * mus_dim_0             (mus_dim_0) int64 16B 0 1
  * weights_dim_0         (weights_dim_0) int64 16B 0 1
  * sigmas_dim_0          (sigmas_dim_0) int64 16B 0 1
  * log_likelihood_dim_0  (log_likelihood_dim_0) int64 4kB 0 1 2 ... 497 498 499
Data variables:
    mus                   (chain, draw, mus_dim_0) float64 128kB -2.291 ... -...
    weights               (chain, draw, weights_dim_0) float64 128kB 0.000862...
    sigmas                (chain, draw, sigmas_dim_0) float64 128kB 2.186 ......
    log_likelihood        (chain, draw, log_likelihood_dim_0) float64 32MB -1...
Attributes:
    created_at:                 2025-09-13T15:09:00.079902+00:00
    arviz_version:              0.22.0
    inference_library:          pymc
    inference_library_version:  5.25.1
    sampling_time:              9.554062366485596
    tuning_steps:               1000